# Padding

## 1. What is Padding?

**Padding** in Natural Language Processing (NLP) is the process of adding extra tokens to sequences so that all sequences have the **same length**.

When text is converted into numerical representations, different sentences usually contain different numbers of tokens:

```text
"I like cars"
→ [12, 45, 89]

"I like autonomous cars"
→ [12, 45, 76, 89]
```

These sequences have different lengths:

* First sequence → 3 tokens
* Second sequence → 4 tokens

Machine learning models typically process data in **batches**, where tensors need compatible dimensions. Padding allows us to make the sequences the same length:

```text
[12, 45, 89,  0]
[12, 45, 76, 89]
```

Here, `0` is used as the **padding token**.

---

## 2. Why Do We Need Padding?

Consider a batch containing three sentences:

```text
"I like cars"
"I like autonomous cars"
"I really like autonomous electric cars"
```

After tokenization:

```text
[12, 45, 89]
[12, 45, 76, 89]
[12, 34, 45, 76, 91, 89]
```

The lengths are:

```text
3
4
6
```

We cannot directly represent these as a regular rectangular tensor because the rows have different lengths.

Padding changes them to:

```text
[12, 45, 89,  0,  0,  0]
[12, 45, 76, 89,  0,  0]
[12, 34, 45, 76, 91, 89]
```

Now every sequence has length `6`.

The resulting tensor can have the shape:

```text
(batch_size, sequence_length)
```

For this example:

```text
(3, 6)
```

This makes the data suitable for batch processing in TensorFlow.

---

## 3. Padding Token

A padding token is a special token added only to make sequences the same length.

A common convention is:

```text
0 = padding
```

For example:

```text
Original:
[15, 27, 43]

Padded:
[15, 27, 43, 0, 0]
```

The value `0` does **not** normally represent a meaningful word. It simply means:

> "There is no actual token at this position."

Depending on the tokenizer and vocabulary, padding may instead be represented by a special token such as:

```text
[PAD]
```

Conceptually:

```text
[PAD] → 0
```

---

## 4. Padding to the Longest Sequence

One common approach is to pad every sequence to the length of the longest sequence in the batch.

For example:

```text
Sequence 1 → 3 tokens
Sequence 2 → 5 tokens
Sequence 3 → 4 tokens
```

Maximum length:

```text
5
```

Therefore:

```text
[12, 45, 89,  0,  0]
[12, 34, 76, 91, 54]
[12, 76, 91, 32,  0]
```

The resulting tensor has:

```text
batch_size = 3
sequence_length = 5
```

---

# 5. Padding in TensorFlow

TensorFlow provides utilities for padding sequences.

A common approach is:

```python
from tensorflow.keras.utils import pad_sequences
```

For example:

```python
from tensorflow.keras.utils import pad_sequences

sequences = [
    [12, 45, 89],
    [12, 45, 76, 89],
    [12, 34, 45, 76, 91, 89]
]

padded_sequences = pad_sequences(
    sequences,
    padding="post"
)

print(padded_sequences)
```

Output:

```text
[[12 45 89  0  0  0]
 [12 45 76 89  0  0]
 [12 34 45 76 91 89]]
```

The longest sequence contains `6` tokens, so all sequences are padded to length `6`.

---

# 6. `padding="post"` vs `padding="pre"`

Padding can be added either at the **end** or at the **beginning** of a sequence.

## Post Padding

```python
pad_sequences(
    sequences,
    padding="post"
)
```

Example:

```text
Original:
[12, 45, 89]

Post padded:
[12, 45, 89, 0, 0]
```

The padding is added **after** the actual tokens.

---

## Pre Padding

```python
pad_sequences(
    sequences,
    padding="pre"
)
```

Example:

```text
Original:
[12, 45, 89]

Pre padded:
[0, 0, 12, 45, 89]
```

The padding is added **before** the actual tokens.

---

## Why Does the Difference Matter?

The position of tokens can matter for sequence models.

For example:

```text
Post padding:

[12, 45, 89, 0, 0]


Pre padding:

[0, 0, 12, 45, 89]
```

With many modern NLP workflows, **post-padding** is commonly used, particularly when working with sequence-processing layers that can efficiently ignore padding.

The important point is to use a padding strategy that is consistent with the model and preprocessing pipeline.

---

# 7. Padding and Truncation

Padding solves the problem of sequences that are **too short**.

But what about sequences that are **too long**?

This is where **truncation** is used.

Suppose we want every sequence to have exactly:

```text
max_length = 5
```

Consider:

```text
[12, 34, 45, 76, 91, 89, 55]
```

This sequence contains 7 tokens.

We need to remove some tokens:

```text
[12, 34, 45, 76, 91]
```

This is called **truncation**.

Therefore:

> **Padding makes short sequences longer. Truncation makes long sequences shorter.**

TensorFlow can perform both operations:

```python
padded_sequences = pad_sequences(
    sequences,
    maxlen=5,
    padding="post",
    truncating="post"
)
```

---

# 8. Example: Padding and Truncation Together

Consider:

```python
sequences = [
    [10, 20, 30],
    [10, 20, 30, 40, 50],
    [10, 20, 30, 40, 50, 60, 70]
]
```

Suppose:

```python
max_length = 5
```

Using:

```python
padded = pad_sequences(
    sequences,
    maxlen=5,
    padding="post",
    truncating="post"
)
```

We get:

```text
[
    [10, 20, 30,  0,  0],
    [10, 20, 30, 40, 50],
    [10, 20, 30, 40, 50]
]
```

Notice what happened:

### Sequence 1

Original:

```text
[10, 20, 30]
```

Too short → padding:

```text
[10, 20, 30, 0, 0]
```

### Sequence 2

Original:

```text
[10, 20, 30, 40, 50]
```

Already the correct length:

```text
[10, 20, 30, 40, 50]
```

### Sequence 3

Original:

```text
[10, 20, 30, 40, 50, 60, 70]
```

Too long → truncation:

```text
[10, 20, 30, 40, 50]
```

---

# 9. Padding and Embeddings

Padding becomes particularly important when using an **Embedding layer**.

Suppose:

```python
padded_sequences = [
    [12, 45, 89, 0, 0],
    [12, 45, 76, 89, 0]
]
```

The sequences can be passed to an embedding layer:

```python
import tensorflow as tf

embedding = tf.keras.layers.Embedding(
    input_dim=10000,
    output_dim=128,
    mask_zero=True
)
```

The output has the conceptual shape:

```text
(batch_size, sequence_length, embedding_dimension)
```

For this example:

```text
(2, 5, 128)
```

The `5` comes from the padded sequence length.

The `128` is the embedding dimension.

---

# 10. Masking Padding

Padding introduces an important problem:

> The model should generally not treat padding tokens as real words.

For example:

```text
[12, 45, 89, 0, 0]
```

The model should understand that:

```text
12 → real token
45 → real token
89 → real token
0  → padding
0  → padding
```

TensorFlow can use **masking** to identify these padding positions.

For example:

```python
embedding = tf.keras.layers.Embedding(
    input_dim=10000,
    output_dim=128,
    mask_zero=True
)
```

With:

```python
mask_zero=True
```

TensorFlow treats index `0` as padding and creates a mask indicating which positions contain real tokens.

Conceptually:

```text
Sequence:
[12, 45, 89, 0, 0]

Mask:
[1,  1,  1,  0, 0]
```

Where:

```text
1 → valid token
0 → padding
```

This allows compatible layers to ignore the padded positions during sequence processing.

---

# 11. Padding with RNNs

Padding is particularly relevant when working with recurrent neural networks such as:

* SimpleRNN
* LSTM
* GRU

For example:

```python
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(
        input_dim=10000,
        output_dim=128,
        mask_zero=True
    ),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(1, activation="sigmoid")
])
```

The workflow becomes:

```text
Text
 ↓
Tokenization
 ↓
Integer sequences
 ↓
Padding
 ↓
Embedding
 ↓
Masking
 ↓
LSTM
 ↓
Dense
 ↓
Prediction
```

---

# 12. Padding and Transformers

Transformers also commonly process batches containing sequences with different lengths.

For example:

```text
Sentence A:
[12, 45, 89]

Sentence B:
[12, 34, 76, 91, 54]
```

They can be padded:

```text
Sentence A:
[12, 45, 89,  0,  0]

Sentence B:
[12, 34, 76, 91, 54]
```

The Transformer needs to know which positions are padding.

This is commonly handled using an **attention mask**.

Conceptually:

```text
Tokens:
[12, 45, 89, 0, 0]

Attention mask:
[ 1,  1,  1, 0, 0]
```

The attention mechanism can then avoid treating the padding positions as meaningful input.

---

# 13. Padding vs Fixed-Length Input

There are two related concepts:

### Padding to the batch maximum

Each batch can have its own maximum length.

For example:

```text
Batch 1 → maximum length = 32
Batch 2 → maximum length = 47
Batch 3 → maximum length = 29
```

This can reduce unnecessary padding.

### Padding to a global maximum

You choose a fixed maximum:

```text
max_length = 128
```

Every sequence becomes length `128`.

```text
Short sequence
→ [tokens + padding]
→ length 128
```

This provides a consistent input shape, but may result in a lot of unnecessary padding.

---

# 14. Why Excessive Padding Is Inefficient

Consider:

```text
Sequence A → 5 tokens
Sequence B → 7 tokens
Sequence C → 500 tokens
```

If all three are padded to `500`:

```text
A → 5 real tokens + 495 padding tokens
B → 7 real tokens + 493 padding tokens
C → 500 real tokens
```

Most of the computation for A and B would involve padding.

This can waste:

* Memory
* GPU/CPU computation
* Training time

Therefore, choosing a reasonable maximum sequence length is an important preprocessing decision.

---

# 15. Padding in a Typical TensorFlow NLP Pipeline

A simplified NLP pipeline looks like this:

```text
Raw Text
   │
   ▼
Tokenization
   │
   ▼
Integer Sequences
   │
   ▼
Padding / Truncation
   │
   ▼
Embedding
   │
   ▼
Sequence Model
   │
   ├── LSTM
   ├── GRU
   └── Transformer
   │
   ▼
Prediction
```

For example:

```python
import tensorflow as tf
from tensorflow.keras.utils import pad_sequences

sequences = [
    [12, 45, 89],
    [12, 45, 76, 89],
    [12, 34, 45, 76, 91, 89]
]

padded = pad_sequences(
    sequences,
    maxlen=6,
    padding="post",
    truncating="post"
)

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(
        input_dim=10000,
        output_dim=128,
        mask_zero=True
    ),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

output = model(padded)
```

---

# 16. Important TensorFlow Parameters

When using `pad_sequences`, several parameters are particularly important:

| Parameter    | Purpose                                       |
| ------------ | --------------------------------------------- |
| `maxlen`     | Maximum sequence length                       |
| `padding`    | Where padding is added: `"pre"` or `"post"`   |
| `truncating` | Where tokens are removed: `"pre"` or `"post"` |
| `value`      | Value used for padding                        |
| `dtype`      | Data type of the resulting sequences          |

Example:

```python
pad_sequences(
    sequences,
    maxlen=100,
    padding="post",
    truncating="post",
    value=0
)
```

This means:

```text
Maximum length = 100
Padding = after the sequence
Truncation = after the sequence
Padding value = 0
```

---

# 17. Padding vs Truncation

| Technique  | Problem Solved                            | Example                       |
| ---------- | ----------------------------------------- | ----------------------------- |
| Padding    | Sequence is too short                     | `[1,2,3] → [1,2,3,0,0]`       |
| Truncation | Sequence is too long                      | `[1,2,3,4,5,6] → [1,2,3,4,5]` |
| Masking    | Prevent padding from affecting processing | `[1,1,1,0,0]`                 |

These three concepts often work together in NLP pipelines.

---

# 18. Key Takeaways

* **Padding** makes variable-length sequences the same length.
* A common padding value is `0`.
* Padding enables efficient **batch processing**.
* Padding can be applied at the beginning (`pre`) or end (`post`) of a sequence.
* **Truncation** handles sequences that are longer than the desired maximum length.
* `tf.keras.utils.pad_sequences()` can perform both padding and truncation.
* Padding tokens should generally not be treated as meaningful input.
* `mask_zero=True` in an embedding layer can be used to identify `0` as padding.
* RNNs, LSTMs, GRUs, and Transformers commonly need mechanisms to handle padded sequences.
* Excessive padding wastes computation and memory.
* Choosing `maxlen` is therefore an important part of NLP preprocessing.

## Mental Model

Think of padding as putting every sentence into a box of the same size:

```text
Short sentence
┌─────────────────────────┐
│ 12 │ 45 │ 89 │ 0 │ 0 │
└─────────────────────────┘

Longer sentence
┌─────────────────────────┐
│ 12 │ 45 │ 76 │ 89 │ 54 │
└─────────────────────────┘
```

The model receives boxes with the same dimensions, while **masking tells the model which parts of the box contain real tokens and which parts are just empty space**.


In [ ]:
import tensorflow as tf
from tensorflow.keras.utils import pad_sequences

# ----------------------------------
# 1. Example tokenized sentences
# ----------------------------------

sequences = [
    [12, 45, 89],
    [12, 45, 76, 89],
    [12, 34, 45, 76, 91, 89]
]

print("#---Original Sequence---#")
for sequence in sequences:
    print(sequence)

# -----------------------------
# 2. Pad the sequences
# -----------------------------

padded_sequences = pad_sequences(
    sequences,
    maxlen=6,
    padding = "post",
    truncating ="post",
    value = 0 
)

print("\nPadded sequences")
print(padded_sequences)


# ---------------------------------
# 3. Creating an Embedding layer
# ---------------------------------

embedding = tf.keras.layers.Embedding(
    input_dim = 100,
    output_dim = 8,
    mask_zero = True
)

# ---------------------------------------
# 4. Convert token IDs into embeddings
# ---------------------------------------

embedded = embedding(padded_sequences)

print(f"\nEmbedding shape: {embedded.shape}")